# 🛒 E-Commerce Data Cleaning Project

This notebook takes the raw file **`Ecommerce_Unclean_Project.xlsx`** and produces a clean, analysis-ready dataset: **`Clean_Ecommerce.csv`**.

**Workflow:**
1. Setup & Imports
2. Load the Raw Data
3. Initial Data Exploration
4. Data Cleaning
5. Feature Engineering
6. Final Review
7. Export Clean Data

## 1. Setup & Imports

Install the required libraries and import the packages used throughout this notebook.

In [ ]:
# Install required libraries
!pip install pandas numpy openpyxl sqlalchemy pymysql

In [ ]:
# Core libraries for data handling
import pandas as pd
import numpy as np

## 2. Load the Raw Data

Read the raw Excel export into a DataFrame.

In [ ]:
df = pd.read_excel('Ecommerce_Unclean_Project.xlsx')

## 3. Initial Data Exploration

Before cleaning anything, get a feel for the dataset: what the rows look like, the column types, summary statistics, size, missing values, and duplicates.

In [ ]:
# Preview the first 5 rows
df.head()

In [ ]:
# Column names, data types, and non-null counts
df.info()

In [ ]:
# Statistical summary of numeric columns
df.describe()

In [ ]:
# Dataset size: (rows, columns)
df.shape

In [ ]:
# Count of missing values per column
df.isnull().sum()

In [ ]:
# Count of fully duplicated rows
df.duplicated().sum()

## 4. Data Cleaning

This is the core cleaning stage. Each step targets one specific data quality issue:

| Step | Issue | Fix |
|---|---|---|
| 4.1 | Duplicate rows | Drop exact duplicates |
| 4.2 | Placeholder nulls (`'N/A'`, `'NULL'`, `''`) | Replace with `np.nan` |
| 4.3 | Inconsistent whitespace | Strip text columns |
| 4.4 | Inconsistent text casing | Convert names/city/state to Title Case |
| 4.5 | Invalid emails | Keep only rows with a valid-looking email |
| 4.6 | Date columns stored as text | Convert to `datetime` |
| 4.7 | Numeric columns stored as text | Convert to numeric |
| 4.8 | Invalid quantities | Remove rows with `Qty <= 0` |
| 4.9 | Remaining missing values | Fill with sensible defaults |


In [ ]:
### 4.1 Remove duplicate rows
df.drop_duplicates(inplace=True)

In [ ]:
### 4.2 Standardize missing-value placeholders to NaN
df.replace(['N/A', 'NULL', ''], np.nan, inplace=True)

In [ ]:
# Quick check after the first cleaning pass
df

In [ ]:
### 4.3 Remove extra leading/trailing whitespace from text columns
df = df.apply(lambda x: x.str.strip() if x.dtype == 'object' else x)

In [ ]:
### 4.4 Standardize text formatting (proper case)
df['Customer_Name'] = df['Customer_Name'].str.title()
df['City'] = df['City'].str.title()
df['State'] = df['State'].str.title()

In [ ]:
### 4.5 Keep only rows with a valid-looking email address
df = df[df["Email"].str.contains("@", na=False)]

In [ ]:
### 4.6 Convert date columns to proper datetime type
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")
df["Delivery_Date"] = pd.to_datetime(df["Delivery_Date"], errors="coerce")

In [ ]:
### 4.7 Convert numeric columns stored as text
cols = ["Qty", "Unit_Price", "Discount"]

for c in cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

In [ ]:
### 4.8 Remove rows with invalid (zero or negative) quantity
df = df[df["Qty"] > 0]

In [ ]:
### 4.9 Fill remaining missing values with sensible defaults
df["Discount"] = df["Discount"].fillna(0)
df["Phone"] = df["Phone"].fillna("Unknown")
df["Delivery_Date"] = df["Delivery_Date"].fillna(df["Order_Date"])

## 5. Feature Engineering

Derive the business metrics and date-based features needed for downstream analysis:

- **Sales** = Qty × Unit Price
- **Net_Amount** = Sales after discount
- **Profit** = 20% margin on Net Amount
- **Month / Year / Weekday** = extracted from Order Date


In [ ]:
# Sales
df["Sales"] = df["Qty"] * df["Unit_Price"]

# Net Amount (after discount)
df["Net_Amount"] = df["Sales"] - (df["Sales"] * df["Discount"] / 100)

# Profit (assumed 20% margin)
df["Profit"] = df["Net_Amount"] * 0.20

# Order Month
df["Month"] = df["Order_Date"].dt.month_name()

# Order Year
df["Year"] = df["Order_Date"].dt.year

# Order Weekday
df["Weekday"] = df["Order_Date"].dt.day_name()

## 6. Final Review

Confirm the cleaned dataset looks correct before exporting.

In [ ]:
# Preview the cleaned dataset
df

In [ ]:
# Confirm final data types and non-null counts
df.info()

## 7. Export Clean Data

Save the cleaned dataset so it can be reused for analysis (e.g. in SQL, Excel, or BI tools).

In [ ]:
# Export the cleaned dataset to CSV
df.to_csv('Clean_Ecommerce.csv', index=False)